In [ ]:
import gseapy as gp 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib import rcParams
from scipy import stats
from gseapy import barplot, dotplot
import os
from gseapy.plot import gseaplot2


In [ ]:
with open('../related_files/TableS2_essential_gene_hart_2017.txt', 'r') as f:
    common_essential = [i.strip() for i in f]
with open('../related_files/CEGs.txt', 'r') as f:
    ogee = [i.strip() for i in f]
df_depmap = pd.read_csv('../related_files/CRISPRInferredCommonEssentials.csv')
depmap_es = df_depmap.iloc[:,1].tolist()
gene_set = {'Hart et al., CEGs':common_essential, 'OGEE CEGs':ogee, 'Depmap CEGs':depmap_es}


for sample in os.listdir('../new_mageck/'):
    if not sample.startswith('.'):
        if sample not in os.listdir('../enrichment/'):
            os.makedirs(f'../enrichment/{sample}')
        gene_summary_path = f'../new_mageck/{sample}/{sample}.gene_summary.txt' 
        df = pd.read_csv(gene_summary_path, sep='\t', index_col=0) 
        df['signed_score'] = df.apply(lambda x: np.log10(x['neg|score']) if x['neg|lfc']<0 else -np.log10(x['pos|score']), axis=1)
        df_hits = df[df['hits'] == -1]

        # Step 1: Define your gene list  
        gene_list = df_hits['human homolog'].astype(str).tolist()  # Example gene list  

        background_genes = df['human homolog'].astype(str).tolist()  # Example background list  

        # Step 2: Perform the KEGG enrichment analysis  
        enrichment = gp.enrichr(  
            gene_list=gene_list,  
            gene_sets=['DisGeNET','ClinVar_2025', 'KEGG_2021_Human', 'Jensen_DISEASES'],
#             background=background_genes,  
            cutoff=0.1,  no_plot = True, outdir = f'../enrichment/{sample}'# Adjust the cutoff value if needed  
        )  
        enrichment_sig = enrichment.results[enrichment.results['Adjusted P-value']<0.1]

        rcParams['font.family'] = 'sans-serif'  
        rcParams['font.sans-serif'] = ['Arial']  
        rcParams['font.size'] = 10
        rcParams['axes.linewidth'] = 0.8  
        rcParams['xtick.major.width'] = 0.8  
        rcParams['ytick.major.width'] = 0.8  
        rcParams['xtick.major.size'] = 3  
        rcParams['ytick.major.size'] = 3  
        rcParams['pdf.fonttype'] = 42  # Ensures text is editable in AI  
        rcParams['ps.fonttype'] = 42  

        for gene_set in ['DisGeNET','KEGG_2021_Human', 'ClinVar_2025', 'Jensen_DISEASES']:
            temp = enrichment_sig[enrichment_sig['Gene_set'] == gene_set]
#             if gene_set == 'KEGG_2021_Human':
            try:  
                top_kegg = temp.head(20)
                top_kegg['-log10 P'] = -np.log10(top_kegg['Adjusted P-value'])
                top_kegg['Term'] = top_kegg['Term'].apply(lambda x: x.split(' (GO')[0])
                top_kegg['Term'] = top_kegg['Term'].apply(lambda x: x[:30] + '...' if len(x) > 30 else x)  
                plt.figure(figsize=(8, 6))  
                if gene_set == 'KEGG_2021_Human':
                    palette = 'viridis_r'
                else:
                    palette = 'magma_r'
                sns.barplot(x='-log10 P', y='Term', data=top_kegg, palette=palette)  
                plt.title(f'Significant {gene_set} Enrichments')  
                plt.xlabel('-log10 Adjusted P-value')  
                plt.ylabel(gene_set)  
                plt.tight_layout()  
                plt.savefig(f'../enrichment/{sample}/{sample}_{gene_set}_enrichment.pdf', bbox_inches = 'tight')  
            except Exception as e:  
                print(f"Could not create plot: {e}")  
                print("Check the column names in the results DataFrame:")  
#             else:
#                 try:
#                     ax2 = dotplot(temp ,cmap='magma_r', top_term = 10, 
#                                   title = 'DisGeNET', cutoff = 0.1, ofname=f'../enrichment/{sample}/{sample}_{gene_set}_enrichment.pdf')
#                 except:
#                     print ('error')
                    
        df.set_index('human homolog', inplace=True)
        gene_set = {'Hart et al., CEGs':common_essential, 'OGEE CEGs':ogee, 'Depmap CEGs':depmap_es}
        try:
            pre_res = gp.prerank(rnk=df.loc[:,'signed_score'], # or rnk = rnk,
                     gene_sets=gene_set,
                     threads=4,
                     outdir=f'../enrichment/{sample}',
                     seed=6,max_size=2000,
                     verbose=True, # see what's going on behind the scenes
                    )
            terms = pre_res.res2d.Term[:]
            hits = [pre_res.results[t]['hits'] for t in terms]
            runes = [pre_res.results[t]['RES'] for t in terms]
            fig = gseaplot2(terms=terms, RESs=runes, hits=hits,
                          legend_kws={'loc': (0, 1)}, # set the legend loc
                          figsize=(8,4), ofname=f'../enrichment/{sample}/{sample}_ceg_GSEA_depmap.pdf') # rank_metric=pre_res.ranking
        except:
            print ('error')